In [ ]:
import os
if os.path.exists('/content/checkmaize'):
    os.system('cd /content/checkmaize && git pull')
else:
    print("OPTION A: push this repo to GitHub and run: !git clone <your-repo-url> /content/checkmaize")
    print("OPTION B (no GitHub): zip the local repo (excluding node_modules, .git, artifacts) and upload to /content/checkmaize.zip, then:")
    print("  !mkdir -p /content/checkmaize && !unzip -q /content/checkmaize.zip -d /content/checkmaize")
print("Then run: !pip install -q -r /content/checkmaize/requirements.txt")


In [ ]:
import os
os.chdir('/content/checkmaize')
from datasets import load_dataset
from PIL import Image
import shutil

dst = 'data/raw/plantvillage'
shutil.rmtree(dst, ignore_errors=True)
CLASS_MAP = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_': 'Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight': 'Northern_Leaf_Blight',
    'Corn_(maize)___healthy': 'healthy',
}
os.makedirs(dst, exist_ok=True)
for split in ['train', 'test']:
    ds = load_dataset('mohanty/PlantVillage', split=split)
    corn = [r for r in ds if r['crop'] == 'Corn (maize)']
    print(f"{split}: {len(corn)} corn rows")
    for r in corn:
        folder = CLASS_MAP[r['label']]
        out = os.path.join(dst, folder, f"{r['leaf_id']}__{r['image_path'].rsplit('/', 1)[-1]}")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        r['image'].save(out)
print("plantvillage extraction done")


In [ ]:
import os, zipfile, shutil
os.chdir('/content/checkmaize')
if not os.path.exists('/content/data/raw'):
    os.makedirs('/content/data/raw', exist_ok=True)
if os.path.exists('/content/Raw Data.zip'):
    with zipfile.ZipFile('/content/Raw Data.zip') as z:
        z.extractall('/content/ccmt_extract')
    src = None
    for root, dirs, files in os.walk('/content/ccmt_extract'):
        if 'Maize' in dirs:
            src = os.path.join(root, 'Maize')
            break
    assert src, "Maize folder not found in upload"
    dst = 'data/raw/ccmt_ghana'
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)
    for name in sorted(os.listdir(dst)):
        print(name, len(os.listdir(os.path.join(dst, name))))
else:
    print("No /content/Raw Data.zip found. Upload it from the Mendeley page "
          "(DOI 10.17632/bwh3zbpkpv.1, download 'Raw Data.zip') using the files pane, then re-run this cell.")


In [ ]:
os.chdir('/content/checkmaize')
!python -m data.make_manifest
!python -m data.make_splits
!python -m pytest data/tests -v


In [ ]:
import shutil, os
shutil.make_archive('/content/splits', 'zip', 'data/manifests')
from google.colab import files
files.download('/content/splits.zip')
